In [ ]:
###############################################################
# QSPR PIPELINE for Anti Cancer Drugs
###############################################################

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import AllChem
from rdkit.Chem import SaltRemover
from rdkit.Chem.MolStandardize import rdMolStandardize

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor

import shap
import matplotlib.pyplot as plt

###############################################################
# LOAD
###############################################################

def load_data(file):

    if file.endswith(".xlsx"):
        df = pd.read_excel(file)
    else:
        df = pd.read_csv(file)

    df = df.drop_duplicates()

    return df


###############################################################
# SMILES STANDARDIZATION & VALIDATION
###############################################################

remover = SaltRemover.SaltRemover()
normalizer = rdMolStandardize.Normalizer()
uncharger = rdMolStandardize.Uncharger()
tautomer = rdMolStandardize.TautomerEnumerator()

allowed_atoms = {
    "C","H","O","N","S","P","F","Cl","Br","I","B","Si"
}

def clean_smiles(s):

    mol = Chem.MolFromSmiles(str(s))

    if mol is None:
        return None

    # remove salts/fragments
    mol = remover.StripMol(mol)

    # normalize
    mol = normalizer.normalize(mol)

    # neutralize
    mol = uncharger.uncharge(mol)

    # tautomer normalization
    mol = tautomer.Canonicalize(mol)

    # rare atom filtering
    for atom in mol.GetAtoms():
        if atom.GetSymbol() not in allowed_atoms:
            return None

    # canonical SMILES
    return Chem.MolToSmiles(mol, canonical=True)


###############################################################
# TOPOLOGICAL DESCRIPTORS
###############################################################

def topo_descriptors(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    A = Chem.GetAdjacencyMatrix(mol)
    n = A.shape[0]

    deg = A.sum(axis=1)

    ###########################################################
    # First Zagreb Index M1
    ###########################################################
    M1 = np.sum(deg**2)

    ###########################################################
    # Distance matrix
    ###########################################################
    D = Chem.GetDistanceMatrix(mol)

    ###########################################################
    # Wiener index
    ###########################################################
    Wiener = 0.5 * np.sum(D)

    ###########################################################
    # Mostar and PI index
    ###########################################################
    Mostar = 0
    PI = 0

    for i in range(n):
        for j in range(i+1,n):

            if A[i,j] == 1:

                di = D[i]
                dj = D[j]

                nu = np.sum(di < dj)
                nv = np.sum(dj < di)

                Mostar += abs(nu - nv)
                PI += (nu + nv)

    return {

        "M1":M1,
        "Wiener":Wiener,
        "Mostar":Mostar,
        "PI":PI

    }


###############################################################
# RDKit DESCRIPTORS
###############################################################

def rdkit_features(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    d = {}

    d["MolWt"] = Descriptors.MolWt(mol)

    d["TPSA"] = rdMolDescriptors.CalcTPSA(mol)

    d["NumRotatableBonds"] = rdMolDescriptors.CalcNumRotatableBonds(mol)

    d["NumHDonors"] = rdMolDescriptors.CalcNumHBD(mol)

    d["NumHAcceptors"] = rdMolDescriptors.CalcNumHBA(mol)

    d["FractionCsp3"] = Descriptors.FractionCSP3(mol)

    d["BalabanJ"] = Descriptors.BalabanJ(mol)

    d["BertzCT"] = Descriptors.BertzCT(mol)

    d["Kappa1"] = Descriptors.Kappa1(mol)

    d["Kappa2"] = Descriptors.Kappa2(mol)

    d["Kappa3"] = Descriptors.Kappa3(mol)

    d["HeavyAtomCount"] = rdMolDescriptors.CalcNumHeavyAtoms(mol)

    return d


###############################################################
# FINGERPRINT
###############################################################

def fingerprint(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return np.zeros(1024)

    return np.array(

        AllChem.GetMorganFingerprintAsBitVect(

            mol,

            3,

            nBits=1024

        )

    )


###############################################################
# FEATURE MATRIX
###############################################################

def build_features(df):

    df["SMILES"] = df.SMILES.apply(clean_smiles)

    df = df.dropna(subset=["SMILES"])

    df = df.drop_duplicates(subset=["SMILES"])

    ###########################################################
    # compute descriptors
    ###########################################################

    rd_df = pd.DataFrame(

        df.SMILES.apply(rdkit_features).tolist()

    )

    topo_df = pd.DataFrame(

        df.SMILES.apply(topo_descriptors).tolist()

    )

    fp = np.vstack(

        df.SMILES.apply(fingerprint)

    )

    fp_df = pd.DataFrame(

        fp,

        columns=[f"FP_{i}" for i in range(1024)]

    )

    ###########################################################
    # ignore non numeric columns automatically
    ###########################################################

    numeric_df = df.select_dtypes(include=np.number)

    X = pd.concat(

        [rd_df, topo_df, numeric_df, fp_df],

        axis=1

    )

    ###########################################################
    # missing descriptor handling
    ###########################################################

    X = X.loc[:, X.isnull().mean() < 0.2]

    imp = SimpleImputer(strategy="median")

    X = pd.DataFrame(

        imp.fit_transform(X),

        columns=X.columns

    )

    X.columns = X.columns.astype(str)

    return X


###############################################################
# FILTER
###############################################################

def filter_features(X):

    vt = VarianceThreshold(0.01)

    X_vt = vt.fit_transform(X)

    X_vt = pd.DataFrame(

        X_vt,

        columns=X.columns[vt.get_support()]

    )

    corr = X_vt.corr().abs()

    upper = corr.where(

        np.triu(np.ones(corr.shape),1).astype(bool)

    )

    drop_cols = [

        c for c in upper.columns

        if any(upper[c] > 0.95)

    ]

    X_vt = X_vt.drop(columns=drop_cols)

    ###########################################################
    # remove extreme outliers caused by descriptor errors
    ###########################################################

    X_vt = X_vt.replace([np.inf,-np.inf],np.nan)

    X_vt = X_vt.dropna(axis=1)

    return X_vt


###############################################################
# METRICS
###############################################################

def Q2(y,y_pred):

    press = np.sum((y-y_pred)**2)

    tss = np.sum((y-np.mean(y))**2)

    return 1 - press/tss


def CCC(y,y_pred):

    y_mean = np.mean(y)

    p_mean = np.mean(y_pred)

    cov = np.mean((y-y_mean)*(y_pred-p_mean))

    return (2*cov)/(

        np.var(y)+np.var(y_pred)

        +(y_mean-p_mean)**2

    )


###############################################################
# SHAP
###############################################################

def shap_select(X,y,n=30):

    m = RandomForestRegressor(random_state=SEED)

    m.fit(X,y)

    sv = shap.Explainer(m,X)(X)

    imp = np.abs(sv.values).mean(0)

    idx = np.argsort(imp)[::-1][:n]

    return X.iloc[:,idx]


###############################################################
# Y RANDOMIZATION
###############################################################

def y_randomization(model,X,y,cv):

    r2_rand = []

    q2_rand = []

    for i in range(30):

        y_perm = np.random.permutation(y)

        preds = []

        obs = []

        for tr,te in cv.split(X):

            sc = StandardScaler()

            Xtr = sc.fit_transform(X.iloc[tr])

            Xte = sc.transform(X.iloc[te])

            model.fit(Xtr,y_perm[tr])

            p = model.predict(Xte)

            preds.extend(p)

            obs.extend(y_perm[te])

        r2_rand.append(

            r2_score(obs,preds)

        )

        q2_rand.append(

            Q2(np.array(obs),np.array(preds))

        )

    return np.mean(r2_rand),np.mean(q2_rand)


###############################################################
# plots
###############################################################

def pred_plot(y,y_pred,path,name,r2,q2):

    os.makedirs(path,exist_ok=True)

    plt.figure()

    plt.scatter(y,y_pred)

    lims=[min(min(y),min(y_pred)),max(max(y),max(y_pred))]

    plt.plot(lims,lims)

    plt.xlabel("Actual")

    plt.ylabel("Predicted")

    plt.title(f"{name}\nR2={round(r2,3)} Q2={round(q2,3)}")

    plt.savefig(

        os.path.join(

            path,

            f"Pred_vs_Actual_{name}.png"

        ),

        dpi=300

    )

    plt.close()


def williams_plot(model,X,y,path,name):

    os.makedirs(path,exist_ok=True)

    p=model.predict(X)

    r=(y-p-np.mean(y-p))/np.std(y-p)

    H=X.values@np.linalg.inv(X.values.T@X.values)@X.values.T

    lev=np.diag(H)

    plt.figure()

    plt.scatter(lev,r)

    plt.axhline(3)

    plt.axhline(-3)

    plt.xlabel("Leverage")

    plt.ylabel("Std residual")

    plt.title(name)

    plt.savefig(

        os.path.join(

            path,

            f"Williams_{name}.png"

        ),

        dpi=300

    )

    plt.close()


###############################################################
# MODELS
###############################################################

models={

"Ridge":Ridge(),

"RF":RandomForestRegressor(random_state=SEED),

"XGB":XGBRegressor(random_state=SEED,verbosity=0)

}


param_grid={

"Ridge":{"alpha":[0.001,0.01,0.1,1,10]},

"RF":{

"n_estimators":[300,500],

"max_depth":[5,10,15]

},

"XGB":{

"learning_rate":[0.01,0.05],

"max_depth":[3,5,7],

"n_estimators":[300,500]

}

}


###############################################################
# MAIN
###############################################################

def run_pipeline(file):

    base=os.path.dirname(file)

    out=os.path.join(base,"QSPR_RESULTS")

    os.makedirs(out,exist_ok=True)

    df=load_data(file)

    X=build_features(df)

    # ==================== FIXED TARGET COLUMN NAMES ====================
    targets = ["logP", "RB", "TPSA", "HeavyAtoms", "pIC50"]
    # ===================================================================

    ###########################################################
    # remove rows with missing target
    ###########################################################

    for t in targets:

        df=df[df[t].notna()]

    results=[]

    best_summary=[]

    for t in targets:

        y=df[t]

        Xt=X.copy()

        for c in [

t,"MolWt","TPSA","NumRotatableBonds"

]:

            if c in Xt.columns:

                Xt=Xt.drop(columns=c)

        Xt=shap_select(filter_features(Xt),y,30)

        cv_outer=KFold(5,shuffle=True,random_state=SEED)

        best_q2=-1

        for name,m in models.items():

            preds_nested=[]

            obs_nested=[]

            for train_ix,test_ix in cv_outer.split(Xt):

                Xtrain,Xtest=Xt.iloc[train_ix],Xt.iloc[test_ix]

                ytrain,ytest=y.iloc[train_ix],y.iloc[test_ix]

                grid=GridSearchCV(

                    m,

                    param_grid[name],

                    cv=3,

                    scoring="r2"

                )

                grid.fit(Xtrain,ytrain)

                model_nested=grid.best_estimator_

                sc=StandardScaler()

                Xtr=sc.fit_transform(Xtrain)

                Xte=sc.transform(Xtest)

                model_nested.fit(Xtr,ytrain)

                p_nested=model_nested.predict(Xte)

                preds_nested.extend(p_nested)

                obs_nested.extend(ytest)

            nested_q2=Q2(np.array(obs_nested),np.array(preds_nested))

            pred=[]

            obs=[]

            for tr,te in cv_outer.split(Xt):

                sc=StandardScaler()

                Xtr=sc.fit_transform(Xt.iloc[tr])

                Xte=sc.transform(Xt.iloc[te])

                grid=GridSearchCV(

                    m,

                    param_grid[name],

                    cv=3,

                    scoring="r2"

                )

                grid.fit(Xt.iloc[tr],y.iloc[tr])

                m_best=grid.best_estimator_

                m_best.fit(Xtr,y.iloc[tr])

                p=m_best.predict(Xte)

                pred.extend(p)

                obs.extend(y.iloc[te])

            pred=np.array(pred)

            obs=np.array(obs)

            cv_r2=r2_score(obs,pred)

            q2=Q2(obs,pred)

            rmse=np.sqrt(mean_squared_error(obs,pred))

            mae=mean_absolute_error(obs,pred)

            final_model=m_best.fit(Xt,y)

            train_pred=final_model.predict(Xt)

            train_r2=r2_score(y,train_pred)

            press=np.sum((obs-pred)**2)

            see=np.sqrt(press/(len(y)-Xt.shape[1]-1))

            adj_r2=1-(1-train_r2)*(len(y)-1)/(len(y)-Xt.shape[1]-1)

            ccc=CCC(obs,pred)

            yr_r2,yr_q2=y_randomization(

                m_best,

                Xt,

                np.array(y),

                cv_outer

            )

            results.append({

"Property":t,

"Model":name,

"R2_train":train_r2,

"R2_CV":cv_r2,

"Q2":q2,

"Nested_CV_Q2":nested_q2,

"RMSE":rmse,

"MAE":mae,

"SEE":see,

"CCC":ccc,

"Adjusted_R2":adj_r2,

"PRESS":press,

"Delta_R2":train_r2-q2,

"n_features":Xt.shape[1],

"Y_random_R2":yr_r2,

"Y_random_Q2":yr_q2

})

            if q2>best_q2:

                best_q2=q2

                best_model=final_model

                best_obs=obs

                best_pred=pred

                best_name=name

        pred_plot(

            best_obs,

            best_pred,

            out,

            f"{t}_{best_name}",

            r2_score(best_obs,best_pred),

            Q2(best_obs,best_pred)

        )

        williams_plot(

            best_model,

            Xt,

            y,

            out,

            f"{t}_{best_name}"

        )

        best_summary.append({

"Property":t,

"Model":best_name,

"R2":r2_score(best_obs,best_pred),

"Q2":Q2(best_obs,best_pred),

"Nested_CV_Q2":nested_q2,

"n_features":Xt.shape[1]

})

    pd.DataFrame(results).to_excel(

        os.path.join(out,"All_models_results.xlsx"),

        index=False

    )

    pd.DataFrame(best_summary).to_excel(

        os.path.join(out,"Best_model_summary.xlsx"),

        index=False

    )

    print("\nSaved in:",out)


###############################################################
# RUN
###############################################################

run_pipeline(

r"D:\Phd\PhD\Papers on topological indices\Drugs screening\Data\pp.xlsx"

)